In [1]:
import torch
import torch._utils
import torch.nn as nn
import torch.optim as optim
import math
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import time

# ==========================================
# 1. THE CNN (Feature Extractor)
# ==========================================
class FaceEmbeddingCNN(nn.Module):
    def __init__(self, embedding_size=128):
        super(FaceEmbeddingCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(128 * 14 * 14, embedding_size)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        x = self.fc(x)
        return nn.functional.normalize(x, p=2, dim=1)

# ==========================================
# 2. ARCFACE LOSS
# ==========================================
class ArcFace(nn.Module):
    def __init__(self, embedding_size, num_classes, margin=0.3, scale=16.0):
        super(ArcFace, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale

    def forward(self, embeddings, labels):
        cosine = nn.functional.linear(nn.functional.normalize(embeddings), nn.functional.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.margin)
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        output = (one_hot * target_logits) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        return output

# ==========================================
# 3. DATA PREPARATION
# ==========================================
DATA_DIR = "/kaggle/input/datasets/inanxr/prosopo-small-dataset/aligned_casia" 

print("--- Starting Data Preparation ---")
print(f"Looking for dataset at: {DATA_DIR}")

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
num_classes = len(dataset.classes)

print(f"✅ Successfully loaded dataset!")
print(f"   -> Total Identities (Classes): {num_classes}")
print(f"   -> Total Images: {len(dataset)}")
print(f"   -> Total Batches per Epoch: {len(dataloader)}\n")

# ==========================================
# 4. TRAINING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Setting up Training on {device.type.upper()} ---")

model = FaceEmbeddingCNN(embedding_size=128).to(device)
arcface = ArcFace(embedding_size=128, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(model.parameters()) + list(arcface.parameters()), lr=0.0001)

epochs = 15 
max_batches = 200 

print(f"Starting training for {epochs} epochs (Capped at {max_batches} batches/epoch)...\n")

for epoch in range(epochs):
    start_time = time.time()
    model.train()
    arcface.train()
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        
        if batch_idx >= max_batches:
            break
            
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        embeddings = model(images)
        outputs = arcface(embeddings, labels)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if (batch_idx + 1) % 20 == 0:
            print(f"   Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{max_batches}] | Current Loss: {loss.item():.4f}")
        
    avg_loss = running_loss / max_batches
    epoch_time = time.time() - start_time
    print(f"✅ End of Epoch [{epoch+1}/{epochs}] | Avg Loss: {avg_loss:.4f} | Time: {epoch_time:.1f}s\n")
    
    torch.save(model.state_dict(), "arcface_public_trained_cnn.pth")

print("--- Saving Final Model ---")
torch.save(model.state_dict(), "arcface_public_trained_cnn.pth")
print("✅ Saved as arcface_public_trained_cnn.pth")

--- Starting Data Preparation ---
Looking for dataset at: /kaggle/input/datasets/inanxr/prosopo-small-dataset/aligned_casia
✅ Successfully loaded dataset!
   -> Total Identities (Classes): 9934
   -> Total Images: 446911
   -> Total Batches per Epoch: 3492

--- Setting up Training on CUDA ---
Starting training for 15 epochs (Capped at 200 batches/epoch)...

   Epoch [1/15] | Batch [20/200] | Current Loss: 14.7836
   Epoch [1/15] | Batch [40/200] | Current Loss: 14.1381
   Epoch [1/15] | Batch [60/200] | Current Loss: 14.2168
   Epoch [1/15] | Batch [80/200] | Current Loss: 14.1176
   Epoch [1/15] | Batch [100/200] | Current Loss: 14.3169
   Epoch [1/15] | Batch [120/200] | Current Loss: 13.8662
   Epoch [1/15] | Batch [140/200] | Current Loss: 13.6168
   Epoch [1/15] | Batch [160/200] | Current Loss: 13.8258
   Epoch [1/15] | Batch [180/200] | Current Loss: 13.8771
   Epoch [1/15] | Batch [200/200] | Current Loss: 13.5839
✅ End of Epoch [1/15] | Avg Loss: 14.0928 | Time: 1527.8s

   Ep